# 07 - Limpieza final y capa oro (de plata a oro)

Este notebook cierra la limpieza que quedó pendiente en la primera entrega y arma la capa oro: tablas pequeñas y ya agregadas de donde salen las respuestas a las preguntas de negocio (notebook 08) y los modelos (notebook 09). Corresponde al numeral 1 de la entrega final (filtros y transformaciones).

Se lee de `/nyc/silver` y se escribe en `/nyc/gold`. Lo que se hace:

1. **Población por barrio**, que no teníamos: sin ella no se pueden calcular tasas en los 55 barrios.
2. **Arrestos**: se marcan los relacionados con el tránsito y se define qué hacer con la edad en 2025.
3. **Choques**: se completan los conteos de víctimas, se agrupan las causas y los tipos de vehículo y se arma una tabla con una fila por choque para el modelo.
4. **Filtros** y **tablas agregadas** por barrio, por borough y por año.

Cada filtro y cada transformación queda anotado con su efecto en la tabla del final.

In [ ]:
import hashlib
import json

import pandas as pd
import requests
from pyspark.sql import functions as F

import geo
from codigos import (ARRESTOS_EMBRIAGUEZ_REGEX, ARRESTOS_TRANSITO_REGEX, BANDERAS_VEHICULO, FRANJAS,
                     GRUPOS_CAUSA, GRUPOS_VEHICULO, SIN_CAUSA, VEHICULO_OTRO)
from config import GOLD, RAW, SILVER, sesion
from estilo import BOROUGHS, miles

spark = sesion("07_capa_oro")

ANIOS_COMPLETOS = list(range(2019, 2026))   # 2026 es parcial y no entra en los promedios anuales
N_ANIOS = len(ANIOS_COMPLETOS)

arrestos = spark.read.parquet(f"{SILVER}/arrestos")
choques = spark.read.parquet(f"{SILVER}/choques")
vehiculos = spark.read.parquet(f"{SILVER}/vehiculos")
pobreza = spark.read.parquet(f"{SILVER}/pobreza")
sat = spark.read.parquet(f"{SILVER}/sat")
condados = spark.read.parquet(f"{SILVER}/poblacion_condados_2018")

N_ARRESTOS, N_CHOQUES = arrestos.count(), choques.count()
print(f"Plata: {N_ARRESTOS:,} arrestos | {N_CHOQUES:,} choques | {vehiculos.count():,} vehículos")

registro = []

def anotar(tipo, tabla, descripcion, efecto):
    """Guarda cada filtro o transformación con su efecto, para la tabla del final."""
    registro.append({"tipo": tipo, "tabla": tabla, "procedimiento": descripcion, "efecto": efecto})
    print(f"[{tipo}] {tabla}: {descripcion} -> {efecto}")

def agrupar(columna, grupos, otro):
    """Devuelve el primer grupo cuyo patrón coincide con el texto; si ninguno coincide, `otro`."""
    expr = F.lit(otro)
    for nombre, patron in reversed(grupos):
        expr = F.when(columna.rlike(patron), nombre).otherwise(expr)
    return expr

## 1. Población por barrio

En la primera entrega solo había población por borough. Para tener tasas en los 55 barrios (PUMA 2010) se suman dos tablas de NYC Open Data, el mismo portal de las demás fuentes:

- **Población por vecindario** (`swpk-hqdp`): población del censo de 2010 de cada uno de los 195 vecindarios (NTA) de la ciudad.
- **Equivalencia entre vecindarios y barrios** (`8ius-dhrr`): dice a qué PUMA pertenece cada vecindario. Los vecindarios se diseñaron como subdivisiones de las PUMA, así que basta con sumar.

Las dos tablas se descargan aquí mismo y se guarda una copia tal cual en la capa cruda. Si la descarga falla, se usa la copia que ya esté en HDFS.

In [ ]:
BASE = "https://data.cityofnewyork.us/resource"

def descargar(nombre, id_fuente):
    ruta = f"{RAW}/{nombre}"
    try:
        r = requests.get(f"{BASE}/{id_fuente}.json", params={"$limit": 50000}, timeout=120)
        r.raise_for_status()
        texto = r.text
        spark.createDataFrame([(texto,)], "value string").coalesce(1).write.mode("overwrite").text(ruta)
        origen = "descargado ahora"
    except requests.RequestException as error:
        print(f"   No se pudo descargar {id_fuente} ({error}). Se usa la copia de {ruta}.")
        texto = "\n".join(fila.value for fila in spark.read.text(ruta).collect())
        origen = "copia en HDFS"
    filas = json.loads(texto)
    print(f"{nombre:<24} {id_fuente}  {len(filas):>5} filas  {origen}  SHA-256 {hashlib.sha256(texto.encode()).hexdigest()[:16]}...")
    return pd.DataFrame(filas)

nta = descargar("poblacion_nta", "swpk-hqdp")
equivalencia = descargar("equivalencia_nta_puma", "8ius-dhrr")

Los vecindarios que no son residenciales (parques, cementerios, aeropuertos) vienen agrupados en un código por borough y quedan repartidos entre varias PUMA, así que no se pueden asignar a una sola. Se listan aparte.

In [ ]:
nta = nta[nta["year"] == "2010"].copy()
nta["poblacion"] = nta["population"].astype(int)

equivalencia = equivalencia.rename(columns={"neighborhood_tabulation_area_nta_code": "nta_code"})
equivalencia["puma"] = equivalencia["puma"].astype(str).str.zfill(4)
assert nta["nta_code"].isin(equivalencia["nta_code"]).all(), "Hay vecindarios sin PUMA en la tabla de equivalencia"

pumas_por_nta = equivalencia.groupby("nta_code")["puma"].nunique()
nta["en_varias_pumas"] = nta["nta_code"].isin(pumas_por_nta[pumas_por_nta > 1].index)

no_asignables = nta[nta["en_varias_pumas"]]
print(f"Vecindarios: {len(nta)} | repartidos en varias PUMA: {len(no_asignables)} "
      f"con {no_asignables['poblacion'].sum():,} habitantes ({100 * no_asignables['poblacion'].sum() / nta['poblacion'].sum():.2f} % de la ciudad)")
no_asignables[["nta_code", "nta_name", "borough", "poblacion"]]

**Verificación cruzada.** La población de 2010 por borough que sale de sumar los vecindarios tiene que coincidir con la que trae la tabla del Departamento de Salud que se obtuvo por scraping en el notebook 06. Son dos fuentes independientes; si no coinciden, el notebook se detiene.

In [ ]:
oficial = (condados.filter(F.col("borough").isNotNull()).select("borough", "poblacion_2010", "poblacion_2018", "area_km2")
           .toPandas().set_index("borough").reindex(BOROUGHS))
suma_nta = nta.groupby("borough")["poblacion"].sum().reindex(BOROUGHS)

verificacion = pd.DataFrame({"vecindarios (NYC Open Data)": suma_nta, "scraping (Depto. de Salud)": oficial["poblacion_2010"]})
verificacion["coincide"] = verificacion.iloc[:, 0] == verificacion.iloc[:, 1]
assert verificacion["coincide"].all(), "La población 2010 por borough no coincide entre las dos fuentes"
verificacion

**Ajuste a 2018.** Las tasas por borough usan la población oficial de 2018. Para que las de barrio sean comparables, la población 2010 de cada barrio se multiplica por un factor de su borough: población oficial 2018 del borough dividida entre la suma 2010 de sus barrios. Así los barrios de cada borough suman exactamente la cifra oficial de 2018, y los pocos habitantes de los vecindarios no asignables quedan repartidos en proporción.

El supuesto es que todos los barrios de un mismo borough crecieron al mismo ritmo entre 2010 y 2018. Es una aproximación: los barrios que crecieron más que su borough quedan con una tasa algo inflada.

Como nombre de cada barrio se usa el de su vecindario más poblado.

In [ ]:
asignables = nta[~nta["en_varias_pumas"]].merge(equivalencia[["nta_code", "puma"]].drop_duplicates(), on="nta_code")
asignables = asignables.sort_values("poblacion", ascending=False)

poblacion_puma = asignables.groupby("puma").agg(
    barrio=("nta_name", "first"),
    vecindarios=("nta_name", lambda s: "; ".join(s)),
    n_vecindarios=("nta_code", "count"),
    poblacion_2010=("poblacion", "sum"),
).reset_index()
poblacion_puma["borough"] = poblacion_puma["puma"].map(geo.borough_de_puma)

factor = oficial["poblacion_2018"] / poblacion_puma.groupby("borough")["poblacion_2010"].sum()
poblacion_puma["poblacion_2018"] = (poblacion_puma["poblacion_2010"] * poblacion_puma["borough"].map(factor)).round().astype(int)

# Tiene que haber exactamente las 55 PUMA del mapa oficial y cuadrar con la población oficial
pumas = geo.cargar_pumas(spark)
assert set(poblacion_puma["puma"]) == set(pumas["puma"]), "Las PUMA con población no son las mismas 55 del mapa"
diferencia = (poblacion_puma.groupby("borough")["poblacion_2018"].sum() - oficial["poblacion_2018"]).abs()
assert (diferencia <= 20).all(), f"Los barrios no suman la población oficial de su borough:\n{diferencia}"

spark.createDataFrame(poblacion_puma[["puma", "barrio", "borough", "vecindarios", "n_vecindarios", "poblacion_2010", "poblacion_2018"]]) \
    .write.mode("overwrite").parquet(f"{SILVER}/poblacion_puma")
anotar("transformación", "población", "población 2010 por vecindario sumada a los 55 barrios y ajustada a 2018 con el factor de su borough",
       f"55 barrios, de {miles(poblacion_puma['poblacion_2018'].min())} a {miles(poblacion_puma['poblacion_2018'].max())} habitantes")

print("Factor 2018/2010 por borough:", factor.round(4).to_dict())
poblacion_puma[["puma", "barrio", "borough", "n_vecindarios", "poblacion_2010", "poblacion_2018"]].sort_values("puma").head(10)

## 2. Arrestos

### 2.1 Arrestos relacionados con el tránsito

La pregunta 8 cruza los arrestos por infracciones de tránsito y por conducir en estado de embriaguez con los choques graves. Se marcan según el tipo de delito (`OFNS_DESC`). La tabla muestra qué textos quedaron marcados, para comprobar que la búsqueda no trae delitos que no son de tránsito.

In [ ]:
delito = F.upper(F.coalesce(F.col("ofns_desc"), F.lit("")))
arrestos = (arrestos
    .withColumn("es_transito", delito.rlike(ARRESTOS_TRANSITO_REGEX))
    .withColumn("es_embriaguez", delito.rlike(ARRESTOS_EMBRIAGUEZ_REGEX)))

marcados = (arrestos.filter("es_transito").groupBy("ofns_desc", "es_embriaguez").count()
            .orderBy(F.desc("count")).toPandas())
anotar("transformación", "arrestos", "marca de arrestos de tránsito y de embriaguez según OFNS_DESC",
       f"{miles(marcados['count'].sum())} de tránsito ({100 * marcados['count'].sum() / N_ARRESTOS:.1f} %), "
       f"de ellos {miles(marcados.loc[marcados['es_embriaguez'], 'count'].sum())} por embriaguez")
marcados

### 2.2 La edad en 2025

En la primera entrega se encontró que el grupo de edad viene vacío en buena parte de los arrestos de 2025. Se revisa por año para decidir.

In [ ]:
edad = (arrestos.groupBy("anio").agg(
            F.count("*").alias("arrestos"),
            F.sum(F.col("grupo_edad").isNull().cast("int")).alias("sin_edad"))
        .orderBy("anio").toPandas())
edad["sin_edad_pct"] = (100 * edad["sin_edad"] / edad["arrestos"]).round(1)
edad

**Decisión:** la edad no se imputa. Si se rellenara con el grupo más frecuente, 2025 quedaría con una distribución de edades inventada. La edad queda vacía donde no viene, y cualquier análisis por edad excluye los años en que falta en más del 5 % de los arrestos. Ninguna de las ocho preguntas de negocio ni los modelos usan la edad, así que esto no afecta lo que sigue.

In [ ]:
anios_sin_edad = edad.loc[edad["sin_edad_pct"] > 5, "anio"].tolist()
anotar("decisión", "arrestos", "la edad vacía no se imputa; los análisis por edad excluyen los años con más de 5 % sin dato",
       f"años excluidos de análisis por edad: {anios_sin_edad}")

## 3. Choques

### 3.1 Conteos de víctimas

El total de heridos y de muertos de cada choque viene en una columna, y además desglosado en peatones, ciclistas y ocupantes de vehículos. Cuando el total viene vacío se reconstruye sumando el desglose. Después se vuelve a calcular si el choque es **grave** (al menos un herido o un muerto).

In [ ]:
def suma_partes(sufijo):
    return sum(F.coalesce(F.col(f"number_of_{quien}_{sufijo}"), F.lit(0)) for quien in ["pedestrians", "cyclist", "motorist"])

ch = (choques
    .withColumn("heridos_partes", suma_partes("injured"))
    .withColumn("muertos_partes", suma_partes("killed"))
    .withColumn("heridos_final", F.coalesce(F.col("heridos"), F.col("heridos_partes")))
    .withColumn("muertos_final", F.coalesce(F.col("muertos"), F.col("muertos_partes")))
    .withColumn("grave_final", (F.col("heridos_final") + F.col("muertos_final")) > 0))

victimas = ch.agg(
    F.sum(F.col("heridos").isNull().cast("int")).alias("heridos_vacio"),
    F.sum(F.col("muertos").isNull().cast("int")).alias("muertos_vacio"),
    F.sum((F.col("heridos").isNull() & (F.col("heridos_partes") > 0)).cast("int")).alias("heridos_recuperados"),
    F.sum((F.col("muertos").isNull() & (F.col("muertos_partes") > 0)).cast("int")).alias("muertos_recuperados"),
    F.sum((F.col("heridos").isNotNull() & (F.col("heridos") != F.col("heridos_partes"))).cast("int")).alias("heridos_total_distinto_del_desglose"),
    F.sum((F.col("grave") != F.col("grave_final")).cast("int")).alias("choques_que_cambian_de_gravedad"),
    F.sum(F.col("grave_final").cast("int")).alias("graves"),
).toPandas()
anotar("transformación", "choques", "total de heridos y muertos reconstruido con el desglose cuando viene vacío; gravedad recalculada",
       f"{int(victimas['heridos_vacio'][0])} totales de heridos vacíos y {int(victimas['muertos_vacio'][0])} de muertos; "
       f"{int(victimas['choques_que_cambian_de_gravedad'][0])} choques cambian de gravedad")
victimas.T.rename(columns={0: "choques"})

Cuando el total y el desglose no coinciden se deja el total, que es el dato principal del reporte. El desglose no cuenta a los usuarios de patinetas y otros medios, así que puede ser menor que el total sin que haya un error.

### 3.2 Causa del choque

El factor contribuyente del primer vehículo es texto con unas 60 variantes. Se agrupa en 12 causas (más "Otra causa" para lo que no encaja) para poder comparar y para que el modelo no tenga categorías con muy pocos casos. `Unspecified`, los vacíos y los códigos numéricos quedan como **Sin especificar**: no se imputan porque no hay forma de saber cuál fue la causa, y adivinarla sesgaría justamente la variable que queremos estudiar.

In [ ]:
factor_1 = F.upper(F.trim(F.col("factor_1")))
sin_causa = factor_1.isNull() | (factor_1 == "") | (factor_1 == "UNSPECIFIED") | factor_1.rlike("^[0-9]+$")
ch = ch.withColumn("causa", F.when(sin_causa, SIN_CAUSA).otherwise(agrupar(factor_1, GRUPOS_CAUSA, "Otra causa")))

causas = (ch.groupBy("causa").agg(
            F.count("*").alias("choques"),
            F.countDistinct(F.col("factor_1")).alias("textos_originales"),
            F.round(100 * F.avg(F.col("grave_final").cast("int")), 1).alias("graves_pct"))
          .orderBy(F.desc("choques")).toPandas())
causas["choques_pct"] = (100 * causas["choques"] / N_CHOQUES).round(1)
causas.to_csv("../figuras/tabla_07_causas.csv", index=False)
anotar("transformación", "choques", f"{int(causas['textos_originales'].sum())} textos de factor contribuyente agrupados en {len(causas)} causas",
       f"{causas.loc[causas['causa'] == SIN_CAUSA, 'choques_pct'].sum():.1f} % sin especificar, "
       f"{causas.loc[causas['causa'] == 'Otra causa', 'choques_pct'].sum():.1f} % en 'Otra causa'")
causas

Para revisar la agrupación, estos son los textos que cayeron en "Otra causa". Si alguno frecuente debería estar en un grupo, se agrega a `GRUPOS_CAUSA` en `codigos.py`.

In [ ]:
ch.filter(F.col("causa") == "Otra causa").groupBy("factor_1").count().orderBy(F.desc("count")).limit(15).toPandas()

### 3.3 Tipo de vehículo

El tipo de vehículo es texto libre con cientos de escrituras. Se agrupa en 10 tipos (más "Otro" y "Sin dato"). Se usan las dos fuentes: la tabla de vehículos (una fila por vehículo) y las cinco columnas de tipo que trae la tabla de choques. Con eso se calcula, para cada choque, cuántos vehículos participaron y si hubo al menos una moto, una bicicleta, un vehículo de micromovilidad eléctrica, un camión, un bus o un taxi.

In [ ]:
def tipo_valido(columna):
    texto = F.upper(F.trim(columna))
    return F.when(texto.isNotNull() & (texto != "") & ~texto.isin("UNKNOWN", "UNK", "UNKNO", "UNKNOWN VEHICLE"), texto)

de_vehiculos = vehiculos.filter(F.col("tiene_choque")).select("collision_id", tipo_valido(F.col("tipo_vehiculo")).alias("tipo"))
de_choques = choques.select("collision_id", F.explode(F.array(*[tipo_valido(F.col(f"tipo_vehiculo_{i}")) for i in range(1, 6)])).alias("tipo"))

tipos = (de_vehiculos.unionByName(de_choques).filter(F.col("tipo").isNotNull())
         .withColumn("grupo", agrupar(F.col("tipo"), GRUPOS_VEHICULO, VEHICULO_OTRO)))

grupos_vehiculo = (de_vehiculos.withColumn("grupo", F.when(F.col("tipo").isNull(), "Sin dato").otherwise(agrupar(F.col("tipo"), GRUPOS_VEHICULO, VEHICULO_OTRO)))
                   .groupBy("grupo").agg(F.count("*").alias("vehiculos"), F.countDistinct("tipo").alias("textos_originales"))
                   .orderBy(F.desc("vehiculos")).toPandas())
grupos_vehiculo["vehiculos_pct"] = (100 * grupos_vehiculo["vehiculos"] / grupos_vehiculo["vehiculos"].sum()).round(1)
grupos_vehiculo.to_csv("../figuras/tabla_07_tipos_vehiculo.csv", index=False)
anotar("transformación", "vehículos", f"{int(grupos_vehiculo['textos_originales'].sum())} textos de tipo de vehículo agrupados en {len(grupos_vehiculo)} tipos",
       f"{grupos_vehiculo.loc[grupos_vehiculo['grupo'] == VEHICULO_OTRO, 'vehiculos_pct'].sum():.1f} % en 'Otro', "
       f"{grupos_vehiculo.loc[grupos_vehiculo['grupo'] == 'Sin dato', 'vehiculos_pct'].sum():.1f} % sin dato")
grupos_vehiculo

In [ ]:
de_vehiculos.filter(F.col("tipo").isNotNull()).withColumn("grupo", agrupar(F.col("tipo"), GRUPOS_VEHICULO, VEHICULO_OTRO)) \
    .filter(F.col("grupo") == VEHICULO_OTRO).groupBy("tipo").count().orderBy(F.desc("count")).limit(15).toPandas()

### 3.4 Una fila por choque para el modelo

Se junta todo en la tabla `choques_modelo`. Además de la causa y los vehículos se agregan dos variables de tiempo: la franja horaria y si fue fin de semana.

La unidad es el choque y no el vehículo. Si se modelara por vehículo, los dos carros de un mismo choque podrían quedar uno en entrenamiento y otro en prueba, y el modelo parecería mejor de lo que es.

In [ ]:
n_tabla = vehiculos.filter(F.col("tiene_choque")).groupBy("collision_id").agg(F.count("*").alias("n_tabla"))
banderas = tipos.groupBy("collision_id").agg(
    *[F.max((F.col("grupo") == grupo).cast("int")).alias(nombre) for nombre, grupo in BANDERAS_VEHICULO.items()])
n_columnas = sum(tipo_valido(F.col(f"tipo_vehiculo_{i}")).isNotNull().cast("int") for i in range(1, 6))

hora = F.col("hora")
franja = (F.when(hora.isNull(), None).when(hora < 6, FRANJAS[0]).when(hora < 12, FRANJAS[1])
           .when(hora < 18, FRANJAS[2]).otherwise(FRANJAS[3]))

modelo = (ch
    .withColumn("n_columnas", n_columnas)
    .join(n_tabla, "collision_id", "left")
    .join(banderas, "collision_id", "left")
    .withColumn("n_vehiculos", F.greatest(F.coalesce(F.col("n_tabla"), F.lit(0)), F.col("n_columnas")))
    .withColumn("franja", franja)
    .withColumn("fin_de_semana", F.col("dia_semana").isin("Sábado", "Domingo").cast("int"))
    .fillna(0, subset=list(BANDERAS_VEHICULO))
    .select("collision_id", "fecha", "anio", "mes", "dia_semana", "hora", "franja", "fin_de_semana",
            "borough", "fuente_borough", "puma", "lat", "lon", "causa", "n_vehiculos", *BANDERAS_VEHICULO,
            F.col("heridos_final").alias("heridos"), F.col("muertos_final").alias("muertos"),
            F.col("grave_final").cast("int").alias("grave"),
            F.col("n_tabla").isNotNull().alias("en_tabla_vehiculos")))

modelo.write.mode("overwrite").parquet(f"{GOLD}/choques_modelo")
modelo = spark.read.parquet(f"{GOLD}/choques_modelo")
assert modelo.count() == N_CHOQUES, "choques_modelo no tiene una fila por choque"

resumen_modelo = modelo.agg(
    F.count("*").alias("choques"),
    F.round(100 * F.avg("grave"), 1).alias("graves_pct"),
    F.round(100 * F.avg(F.col("en_tabla_vehiculos").cast("int")), 1).alias("con_filas_en_vehiculos_pct"),
    F.round(F.avg("n_vehiculos"), 2).alias("vehiculos_promedio"),
    F.sum((F.col("n_vehiculos") == 0).cast("int")).alias("sin_ningun_vehiculo"),
    *[F.round(100 * F.avg(b), 1).alias(f"{b}_pct") for b in BANDERAS_VEHICULO],
).toPandas()
anotar("transformación", "choques", "franja horaria, fin de semana, número de vehículos y presencia de moto, bicicleta, micromovilidad, camión, bus y taxi",
       f"tabla choques_modelo con {miles(N_CHOQUES)} filas, una por choque; {resumen_modelo['graves_pct'][0]} % graves")
resumen_modelo.T.rename(columns={0: "valor"})

## 4. Filtros

Para las tasas y los modelos se aplican estos filtros. Ninguno borra datos de la capa plata: solo definen qué entra en cada tabla agregada.

1. **Años completos (2019 a 2025)** para los promedios anuales. El primer semestre de 2026 se conserva para ver la tendencia, pero no entra en los promedios porque medio año no es comparable con uno entero.
2. **Registros con barrio**: los que no tienen coordenadas válidas no se pueden ubicar en una PUMA y quedan por fuera de los análisis por barrio (sí entran en los de borough cuando traen el borough).
3. **Choques con borough** para el modelo, porque el borough es una de sus variables.

In [ ]:
def embudo(tabla, df):
    total = df.count()
    completos = df.filter(F.col("anio").isin(ANIOS_COMPLETOS))
    n_completos = completos.count()
    n_barrio = completos.filter(F.col("puma").isNotNull()).count()
    n_borough = completos.filter(F.col("borough").isNotNull()).count()
    anotar("filtro", tabla, "años completos 2019 a 2025 (para promedios anuales)",
           f"{miles(total)} -> {miles(n_completos)} ({100 * n_completos / total:.1f} %)")
    anotar("filtro", tabla, "con barrio (coordenadas válidas dentro de una PUMA)",
           f"{miles(n_completos)} -> {miles(n_barrio)} ({100 * n_barrio / n_completos:.1f} %)")
    anotar("filtro", tabla, "con borough",
           f"{miles(n_completos)} -> {miles(n_borough)} ({100 * n_borough / n_completos:.1f} %)")

embudo("arrestos", arrestos)
embudo("choques", modelo)

## 5. Tablas agregadas

### 5.1 Por barrio y año, y por borough y año

In [ ]:
def es(columna, valor):
    return F.sum((F.col(columna) == valor).cast("int"))

metricas_arrestos = [
    F.count("*").alias("arrestos"),
    es("gravedad", "Felonía").alias("felonias"),
    es("gravedad", "Delito menor").alias("delitos_menores"),
    F.sum(F.col("es_transito").cast("int")).alias("arrestos_transito"),
    F.sum(F.col("es_embriaguez").cast("int")).alias("arrestos_embriaguez"),
]
metricas_choques = [
    F.count("*").alias("choques"),
    F.sum("grave").alias("choques_graves"),
    F.sum("heridos").alias("heridos"),
    F.sum("muertos").alias("muertos"),
]

arrestos_puma_anio = arrestos.filter(F.col("puma").isNotNull()).groupBy("puma", "anio").agg(*metricas_arrestos)
choques_puma_anio = modelo.filter(F.col("puma").isNotNull()).groupBy("puma", "anio").agg(*metricas_choques)
arrestos_borough_anio = arrestos.filter(F.col("borough").isNotNull()).groupBy("borough", "anio").agg(*metricas_arrestos)
choques_borough_anio = modelo.filter(F.col("borough").isNotNull()).groupBy("borough", "anio").agg(*metricas_choques)

for nombre, df in [("arrestos_puma_anio", arrestos_puma_anio), ("choques_puma_anio", choques_puma_anio),
                   ("arrestos_borough_anio", arrestos_borough_anio), ("choques_borough_anio", choques_borough_anio)]:
    df.coalesce(1).write.mode("overwrite").parquet(f"{GOLD}/{nombre}")
    print(f"{nombre:<24} {spark.read.parquet(f'{GOLD}/{nombre}').count():>5} filas")

# Tipos de delito por borough, para la pregunta 3
(arrestos.filter(F.col("borough").isNotNull()).groupBy("borough", "ofns_desc", "gravedad").count()
 .coalesce(1).write.mode("overwrite").parquet(f"{GOLD}/arrestos_delito_borough"))

### 5.2 Los 55 barrios

Una fila por barrio con sus tasas anuales por cada 100.000 habitantes (promedio de 2019 a 2025). Esta tabla responde las preguntas por barrio y es la entrada del modelo no supervisado.

In [ ]:
def promedio_anual(ruta, llave):
    df = spark.read.parquet(f"{GOLD}/{ruta}").filter(F.col("anio").isin(ANIOS_COMPLETOS))
    columnas = [c for c in df.columns if c not in (llave, "anio")]
    return df.groupBy(llave).agg(*[(F.sum(c) / N_ANIOS).alias(c) for c in columnas]).toPandas().set_index(llave)

def agregar_tasas(tabla):
    """Tasas por 100.000 habitantes al año y porcentajes, a partir de los promedios anuales."""
    por_100k = 1e5 / tabla["poblacion_2018"]
    tabla["tasa_arrestos"] = tabla["arrestos"] * por_100k
    tabla["tasa_felonias"] = tabla["felonias"] * por_100k
    tabla["tasa_delitos_menores"] = tabla["delitos_menores"] * por_100k
    tabla["tasa_arrestos_transito"] = tabla["arrestos_transito"] * por_100k
    tabla["tasa_arrestos_embriaguez"] = tabla["arrestos_embriaguez"] * por_100k
    tabla["tasa_choques"] = tabla["choques"] * por_100k
    tabla["tasa_choques_graves"] = tabla["choques_graves"] * por_100k
    tabla["tasa_heridos"] = tabla["heridos"] * por_100k
    tabla["pct_choques_graves"] = 100 * tabla["choques_graves"] / tabla["choques"]
    tabla["pct_felonias"] = 100 * tabla["felonias"] / tabla["arrestos"]
    return tabla

barrios = (poblacion_puma.set_index("puma")[["barrio", "borough", "poblacion_2010", "poblacion_2018"]]
           .join(promedio_anual("arrestos_puma_anio", "puma"))
           .join(promedio_anual("choques_puma_anio", "puma")))
assert barrios.notna().all().all(), "Hay barrios sin arrestos o sin choques: revisar el cruce espacial"
barrios = agregar_tasas(barrios).reset_index()

spark.createDataFrame(barrios).coalesce(1).write.mode("overwrite").parquet(f"{GOLD}/barrios")
barrios.round(1).to_csv("../figuras/tabla_07_barrios.csv", index=False)
barrios[["puma", "barrio", "borough", "poblacion_2018", "tasa_arrestos", "tasa_choques_graves", "pct_choques_graves"]] \
    .sort_values("tasa_arrestos", ascending=False).round(1).head(10)

### 5.3 Los 5 boroughs

Las mismas tasas por borough, más las variables de contexto: pobreza y educación (encuesta de 2018, ponderada por `PWGTP`) y puntaje del SAT de 2012. De aquí en adelante las tasas por borough usan la **población oficial de 2018** del scraping, no la de la encuesta. Por eso bajan un poco frente a las de la primera entrega; el notebook 06 ya había mostrado que el orden de los boroughs no cambia.

In [ ]:
peso = F.col("peso_persona")
contexto = (pobreza.groupBy("borough").agg(
                (100 * F.sum(F.when(F.col("en_pobreza"), peso).otherwise(0)) / F.sum(peso)).alias("pobreza_pct"),
                (100 * F.sum(F.when((F.col("edad") >= 18) & (F.col("educacion_codigo") == 4), peso).otherwise(0))
                     / F.sum(F.when((F.col("edad") >= 18) & F.col("educacion_codigo").isNotNull(), peso).otherwise(0))).alias("pregrado_pct"),
                (100 * F.sum(F.when((F.col("edad") >= 18) & (F.col("educacion_codigo") == 1), peso).otherwise(0))
                     / F.sum(F.when((F.col("edad") >= 18) & F.col("educacion_codigo").isNotNull(), peso).otherwise(0))).alias("sin_secundaria_pct"))
            .toPandas().set_index("borough"))
sat_borough = sat.groupBy("borough").agg(F.avg("total").alias("sat_total")).toPandas().set_index("borough")

boroughs = (oficial[["poblacion_2018", "area_km2"]]
            .join(contexto).join(sat_borough)
            .join(promedio_anual("arrestos_borough_anio", "borough"))
            .join(promedio_anual("choques_borough_anio", "borough")))
assert boroughs.notna().all().all(), "Falta algún dato de contexto por borough"
boroughs = agregar_tasas(boroughs).reset_index()

spark.createDataFrame(boroughs).coalesce(1).write.mode("overwrite").parquet(f"{GOLD}/boroughs")
boroughs.round(1).to_csv("../figuras/tabla_07_boroughs.csv", index=False)
boroughs[["borough", "poblacion_2018", "pobreza_pct", "pregrado_pct", "sat_total", "tasa_arrestos", "tasa_felonias",
          "tasa_choques_graves", "pct_choques_graves"]].round(1)

## 6. Verificaciones

Antes de usar la capa oro se comprueba que los agregados cuadran con la capa plata: los totales por barrio y por borough tienen que ser iguales a los conteos directos.

In [ ]:
def total(ruta, columna):
    return spark.read.parquet(f"{GOLD}/{ruta}").agg(F.sum(columna)).first()[0]

verificaciones = {
    "arrestos por barrio = arrestos con PUMA": total("arrestos_puma_anio", "arrestos") == arrestos.filter(F.col("puma").isNotNull()).count(),
    "arrestos por borough = arrestos con borough": total("arrestos_borough_anio", "arrestos") == arrestos.filter(F.col("borough").isNotNull()).count(),
    "choques por barrio = choques con PUMA": total("choques_puma_anio", "choques") == modelo.filter(F.col("puma").isNotNull()).count(),
    "choques por borough = choques con borough": total("choques_borough_anio", "choques") == modelo.filter(F.col("borough").isNotNull()).count(),
    "55 barrios": len(barrios) == 55,
    "5 boroughs": len(boroughs) == 5,
    "población de los barrios = población oficial de la ciudad (±100)": abs(barrios["poblacion_2018"].sum() - oficial["poblacion_2018"].sum()) <= 100,
}
for nombre, ok in verificaciones.items():
    print(("OK   " if ok else "FALLA"), nombre)
assert all(verificaciones.values()), "La capa oro no cuadra con la capa plata"

## 7. Resumen de filtros y transformaciones

In [ ]:
pd.set_option("display.max_colwidth", 200)
registro_df = pd.DataFrame(registro)
registro_df.to_csv("../figuras/tabla_07_filtros_transformaciones.csv", index=False)
registro_df

## Conclusiones

- La capa oro tiene siete tablas: `choques_modelo` (una fila por choque), cuatro agregadas por barrio o borough y año, `arrestos_delito_borough`, y las tablas finales `barrios` (55) y `boroughs` (5).
- Ya hay población por barrio. Se validó contra una fuente independiente y se ajustó a 2018 con un supuesto que hay que mencionar: crecimiento uniforme dentro de cada borough.
- Se aplicaron tres filtros (años completos, registros con barrio, registros con borough) y cinco transformaciones (población por barrio, marca de arrestos de tránsito, conteos de víctimas, grupos de causa y de vehículo, variables del choque).
- No se imputó nada que pudiera sesgar el análisis: la edad vacía y la causa sin especificar se conservan como tales.